In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries loaded successfully!")

In [ ]:
data_file = 'marketing_campaign.csv'

try:
    df = pd.read_csv(data_file, sep='\t')
    print("Data loaded successfully!")
    print(f"Dataset shape: {df.shape[0]} rows and {df.shape[1]} columns.")
except FileNotFoundError:
    print("File not found. Please place 'marketing_campaign.csv' in the current working directory.")

df.head()

In [ ]:
df['Income'] = df['Income'].fillna(df['Income'].median())

current_year = 2026
df['Age'] = current_year - df['Year_Birth']

spending_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df['Total_Spending'] = df[spending_cols].sum(axis=1)

df['Children'] = df['Kidhome'] + df['Teenhome']

features = ['Income', 'Age', 'Total_Spending', 'Children', 'Recency', 'NumWebVisitsMonth']
X = df[features].copy()

X = X[(X['Age'] < 100) & (X['Income'] < 200000)]

print("Selected features preview:")
X.head()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Data scaling complete!")

In [ ]:
inertia_values = []
k_values = range(1, 10)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_values_)

plt.figure(figsize=(8, 4))
plt.plot(k_values, inertia_values, marker='o', color='b')
plt.title('Elbow Curve')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

sil_scores = []
k_values = range(2, 10)

for k in k_values:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans_temp.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, cluster_labels)
    sil_scores.append(score)
    print(f"For K = {k}, Silhouette Score = {score:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(k_values, sil_scores, marker='o', color='green')
plt.title('Silhouette Analysis')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
X['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

print("K-Means Cluster Counts:")
print(X['KMeans_Cluster'].value_counts())

plt.figure(figsize=(8, 5))
sns.scatterplot(
    x='Income',
    y='Total_Spending',
    hue='KMeans_Cluster',
    data=X,
    palette='Set1',
    s=70,
    alpha=0.8
)
plt.title('K-Means Clusters: Income vs Total Spending')
plt.xlabel('Income ($)')
plt.ylabel('Total Spending ($)')
plt.show()

In [ ]:
dbscan = DBSCAN(eps=1.2, min_samples=5)
X['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)

print("DBSCAN Cluster Counts (-1 represents noise/outliers):")
print(X['DBSCAN_Cluster'].value_counts())

plt.figure(figsize=(8, 5))
sns.scatterplot(
    x='Income',
    y='Total_Spending',
    hue='DBSCAN_Cluster',
    data=X,
    palette='Set2',
    s=70,
    alpha=0.8
)
plt.title('DBSCAN Clusters: Income vs Total Spending (-1 = Outliers)')
plt.xlabel('Income ($)')
plt.ylabel('Total Spending ($)')
plt.show()

In [ ]:
pca = PCA(n_components=2)
pca_res = pca.fit_transform(X_scaled)
X['PCA1'], X['PCA2'] = pca_res[:, 0], pca_res[:, 1]

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_res = tsne.fit_transform(X_scaled)
X['tSNE1'], X['tSNE2'] = tsne_res[:, 0], tsne_res[:, 1]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(x='PCA1', y='PCA2', hue='KMeans_Cluster', data=X, palette='Set1', ax=axes[0, 0])
axes[0, 0].set_title('K-Means via PCA')

sns.scatterplot(x='PCA1', y='PCA2', hue='DBSCAN_Cluster', data=X, palette='Set2', ax=axes[0, 1])
axes[0, 1].set_title('DBSCAN via PCA')

sns.scatterplot(x='tSNE1', y='tSNE2', hue='KMeans_Cluster', data=X, palette='Set1', ax=axes[1, 0])
axes[1, 0].set_title('K-Means via t-SNE')

sns.scatterplot(x='tSNE1', y='tSNE2', hue='DBSCAN_Cluster', data=X, palette='Set2', ax=axes[1, 1])
axes[1, 1].set_title('DBSCAN via t-SNE')

plt.tight_layout()
plt.show()

In [ ]:
cluster_summary = X.groupby('KMeans_Cluster')[['Income', 'Age', 'Total_Spending', 'Children', 'Recency', 'NumWebVisitsMonth']].mean()
print("Customer Profile Summaries (Mean Values):")
cluster_summary.round(2)